# Laboratório integrado — Detecção de mixers Ethereum, auditorias e gráficos

**Interface única para estudar e demonstrar EXP-01J, EXP-02A e EXP-07.** Importa os resultados oficiais congelados do Google Drive, verifica hashes, explica a validade e cria gráficos em alta resolução. Os notebooks originais continuam preservados para a reprodução detalhada dos EXP-01 a EXP-07. **Executar Tudo** neste painel não equivale a reprocessar todos os dumps da Ethereum.

**Alertas não são rótulos.** Em 2026 foram nove alertas (três Tornado conhecidos e seis indeterminados); há um único contrato positivo nas 23 linhas do treino. Não se estima precision ou F1 populacional para a rede.

**Inspiração visual:** fluxos de metodologia semelhantes ao estudo Leale e Sendin (SBSeg 2025); diagramas de pipeline e visualização de padrões comparáveis em estilo (não em métricas nem escopo) aos artigos GNN-based Detection... e MixGuard. Nada foi copiado das figuras originais.

In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn pyarrow google-api-python-client

## 1. Carregar entradas científicas e conferir integridade

Usa a autorização Google Drive do próprio Colab. Não inserir tokens pessoais, não disponibilizar as planilhas cegas EXP-05 e não modificar os dados de entrada.

In [ ]:
from pathlib import Path
import os, io, json, hashlib, pandas as pd, numpy as np, matplotlib.pyplot as plt
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
from IPython.display import display
from datetime import datetime,timezone

auth.authenticate_user()
service=build('drive','v3')
ROOT=Path('/content/lab_ethereum');ROOT.mkdir(exist_ok=True)
FIG=ROOT/'figures';FIG.mkdir(exist_ok=True)
IDS={
 'manifesto_exp07.json':'1rAwtsQbsX5rhmgI-OLn8GBmJxy0rgGAo',
 'auditoria_matriz.json':'1Ghgugyo8U0Mhn4GAsZlaOmnJ5Clq4nUe',
 'features_treino_original.parquet':'1_Sry0K0JusRL3PcxEweyC_9hnaP1vdRy',
 'folds_congelados.json':'1cg1fkkmFudgY-eqej__5euY_gI5Yac_g',
 'metricas_12_combinacoes_por_fold.csv':'1TB1oy9ZwzYL_EMggYTvZqY3ENHXZj0Uh',
 'resumo_12_combinacoes.csv':'1h8dC2pmki_TteMGXKLHaVAEmVK8duKTO',
 'sinalizados_teste_2026.csv':'11kWohs3VvdDGVjwsN_t-S8EQVix6n9mf',
 'comparacao_elegibilidade.csv':'1fGYC9pNVHlGILcEbJnlqBqulEWZTM0-o',
}
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def obter(name,id):
    path=ROOT/name
    if path.exists():return path
    req=service.files().get_media(fileId=id)
    b=io.BytesIO();down=MediaIoBaseDownload(b,req);done=False
    while not done:_,done=down.next_chunk()
    path.write_bytes(b.getvalue());return path
for fn,fid in IDS.items():obter(fn,fid)
manifesto=json.loads((ROOT/'manifesto_exp07.json').read_text())
checks={f:sha(ROOT/f)==expected for f,expected in manifesto['arquivos_sha256'].items()}
assert all(checks.values()),f'Integridade EXP-07 divergente: {checks}'
print('Integridade oficial EXP-07 OK:',len(checks),'arquivos SHA-256 conferidos')
print('Fonte da amostra janeiro 2026: arquivo oficial Drive; não é ground truth externo')
AUD=json.loads((ROOT/'auditoria_matriz.json').read_text())
print('Auditoria da matriz:',AUD['class_counts'],'contratos por classe:',AUD['contracts_by_class'])
assert AUD['contracts_by_class'].get('1')==1

## 2. Figuras de reprodução e sensibilidade (contagens auditadas)

As contagens 63, 9, 49, 14, 9 e 7 vêm da reprodução final e do protocolo EXP-01J. Os dois tipos de gráfico mostram **quantos alertas** foram produzidos, não quantos são corretos. A matriz CSV com resumo EXP-07 é baixada e verificável por hash.

In [ ]:
plt.rcParams.update({'figure.figsize':(8.5,4.5),'figure.dpi':120,'savefig.dpi':250,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
PETROL='#176B78';GRAY='#64748B';ORANGE='#AA5935'
def export(nome):
    plt.tight_layout();p=FIG/nome;plt.savefig(p,bbox_inches='tight');plt.show();print('Figura salva:',p)
fig,ax=plt.subplots()
labels=['Artigo SBSeg 2025','Reprodução consolidada'];vals=[63,9]
ax.barh(labels[::-1],vals[::-1],color=[PETROL,GRAY],height=.56)
for y,v in enumerate(vals[::-1]):ax.text(v+.8,y,str(v),va='center',fontweight='bold')
ax.set(xlim=(0,73),xlabel='Quantidade de alertas — 29/10/2020',title='Reprodução: diferença na contagem de alertas')
ax.text(.99,.02,'Pools Tornado Cash identificados: 3/3 em ambos',transform=ax.transAxes,ha='right',fontsize=10)
export('01_comparacao_alertas_2020.png')

fig,ax=plt.subplots();xs=[1,5,10,20];ys=[49,14,9,7]
ax.plot(xs,ys,'o-',color=PETROL,linewidth=2.5,markersize=8)
for x,y in zip(xs,ys):ax.annotate(str(y),(x,y),xytext=(0,8),textcoords='offset points',ha='center',fontweight='bold')
ax.set(xticks=xs,xlabel='Mínimo de transações por endereço',ylabel='Total de alertas',title='EXP-01J — Sensibilidade do filtro min_tx')
ax.grid(alpha=.2);export('02_sensibilidade_min_tx.png')

## 3. Figura 3 para o artigo — comparação dos modelos (EXP-07)

Heatmap exportado como `fig03_heatmap_exp07.png` por script Python reproduzível. O CSV de 12 combinações é conferido por SHA-256 e a figura usa **F1 médio interno** nos cinco folds **por linhas**. O mesmo contrato positivo aparece no treino e na validação em todos os folds; portanto, **não demonstra generalização**.


In [ ]:
# Figura 3 do artigo: mesmo script auditável do repositório, sem alterar dados ou refazer treinamento.
import subprocess, sys
from IPython.display import Image, display
from pathlib import Path
SCRIPT_HEATMAP = ROOT/'gerar_fig03_heatmap_exp07.py'
SCRIPT_HEATMAP.write_text('#!/usr/bin/env python3\n"""Figura 3 -- mapa de calor do EXP-07 com o resumo original auditado.\n\nReprodução no repositório/Colab:\n  python scripts/figures/gerar_fig03_heatmap_exp07.py \\\n    --input resumo_12_combinacoes.csv --output fig03_heatmap_exp07.png\n\nA figura retrata médias de F1 dos cinco folds estratificados POR LINHAS.\nComo há um único contrato positivo, NÃO é um teste de generalização.\nNão cria resultados, não reentreina modelos e não infere rótulos novos.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport matplotlib.colors as mcolors\nimport matplotlib.patheffects as pe\nimport numpy as np\nimport pandas as pd\n\nSHA256_RESUMO = \'f1b2da3a9916c298a3d0731b781955ec05c7defd2f83455163fcf2eb68dda6e6\'\nMODELS = {\n    \'RandomForest\': \'Random Forest\',\n    \'DecisionTree\': \'Árvore de decisão\',\n    \'GradientBoosting\': \'Gradient Boosting\',\n    \'RegressaoLogistica\': \'Regressão logística\',\n}\nSETS = {\n    \'A_original_5\': \'A — Original\\n5 atributos\',\n    \'B_sem_volume_4\': \'B — Sem volume\\n4 atributos\',\n    \'C_entropias_3\': \'C — Entropias\\n3 atributos\',\n}\n\n\ndef make_figure(input_path: Path, output_path: Path, dpi: int = 350):\n    if not input_path.is_file():\n        raise FileNotFoundError(input_path)\n    checksum = hashlib.sha256(input_path.read_bytes()).hexdigest()\n    if checksum != SHA256_RESUMO:\n        raise ValueError(f\'SHA-256 inválido. Recebido {checksum}; esperado {SHA256_RESUMO}\')\n    d = pd.read_csv(input_path)\n    must = {\'configuracao\', \'modelo\', \'f1_mean\', \'f1_std\'}\n    if not must.issubset(d.columns):\n        raise ValueError(f\'Colunas faltantes: {sorted(must-set(d.columns))}\')\n    if len(d) != 12 or d[[\'configuracao\',\'modelo\']].duplicated().any():\n        raise ValueError(\'Esperadas exatamente 12 combinações únicas (4 modelos x 3 grupos).\')\n    if set(d.modelo) != set(MODELS) or set(d.configuracao) != set(SETS):\n        raise ValueError(\'Modelos ou configurações não correspondem ao protocolo EXP-07 v2.\')\n    vals = d.pivot(index=\'modelo\', columns=\'configuracao\', values=\'f1_mean\')\n    vals = vals.loc[list(MODELS), list(SETS)].to_numpy(dtype=float)\n    if not np.all(np.isfinite(vals)) or not np.all((vals >= 0) & (vals <= 1)):\n        raise ValueError(\'F1 deve ser numérico entre zero e um.\')\n\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\', \'font.size\':10,\n                         \'axes.spines.top\':False, \'axes.spines.right\':False})\n    fig = plt.figure(figsize=(9.8, 5.0), facecolor=\'white\')\n    ax = fig.add_axes([0.22, 0.255, 0.68, 0.475])\n    cmap = plt.get_cmap(\'YlGnBu\')\n    norm = mcolors.Normalize(vmin=0.50, vmax=1.00)\n    im = ax.imshow(vals, cmap=cmap, norm=norm, aspect=\'auto\', interpolation=\'nearest\')\n    ax.set_xticks(range(3), [SETS[s] for s in SETS], fontsize=11, fontweight=\'medium\')\n    ax.set_yticks(range(4), [MODELS[m] for m in MODELS], fontsize=10.5)\n    ax.tick_params(length=0, pad=10)\n    ax.xaxis.tick_top()\n    for s in ax.spines.values(): s.set_visible(False)\n    ax.set_xticks(np.arange(-.5,3,1), minor=True)\n    ax.set_yticks(np.arange(-.5,4,1), minor=True)\n    ax.grid(which=\'minor\', color=\'white\', linewidth=3)\n    ax.tick_params(which=\'minor\', bottom=False, left=False)\n\n    for i in range(4):\n        for j in range(3):\n            x = vals[i,j]\n            rgba = cmap(norm(x))\n            lum = (0.2126*rgba[0]+0.7152*rgba[1]+0.0722*rgba[2])\n            txt_color = \'#FFFFFF\' if lum < 0.55 else \'#213346\'\n            ax.text(j, i, f\'{x:.4f}\'.replace(\'.\',\',\'), ha=\'center\', va=\'center\',\n                    color=txt_color, fontsize=14.5, fontweight=\'bold\' if i == 0 else \'medium\')\n\n    bar = fig.colorbar(im, ax=ax, orientation=\'vertical\', fraction=.036, pad=.023,\n                       shrink=0.96, ticks=[0.5,0.6,0.7,0.8,0.9,1.0])\n    bar.ax.tick_params(labelsize=8.5, length=0)\n    bar.ax.set_yticklabels([\'0,5\',\'0,6\',\'0,7\',\'0,8\',\'0,9\',\'1,0\'])\n    bar.set_label(\'F1 médio interno\', fontsize=9.5, labelpad=12)\n    bar.outline.set_visible(False)\n\n    fig.text(.055, .962, \'Comparação de classificadores e ablação — EXP-07\',\n             fontsize=16, weight=\'bold\', color=\'#1E334A\', va=\'top\')\n    fig.text(.055, .903, \'12 combinações · cinco folds estratificados por linhas · mesma matriz rotulada\',\n             fontsize=10.5, color=\'#4B5F71\', va=\'top\')\n    fig.add_artist(plt.Line2D([.055,.945],[.185,.185], transform=fig.transFigure,\n                              color=\'#D7E1E9\', linewidth=1.0))\n    fig.text(.055, .142, \'AVALIAÇÃO EXPLORATÓRIA\', fontsize=9.7,\n             color=\'#9D603B\', weight=\'bold\', va=\'center\')\n    fig.text(.055, .095,\n             \'Os 23 positivos pertencem a um único contrato, presente no treino e na validação dos cinco folds.\',\n             fontsize=9.65, color=\'#3A4758\', va=\'center\')\n    fig.text(.055, .055, \'F1 interno não demonstra detecção de contratos ou protocolos inéditos.\',\n             fontsize=9.55, color=\'#3A4758\', va=\'center\')\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(output_path, dpi=dpi, facecolor=\'white\', bbox_inches=\'tight\', pad_inches=.14)\n    plt.close(fig)\n    print(\'PNG:\', output_path.resolve())\n    print(\'SHA-256 CSV:\', checksum)\n    print(\'Matriz F1 (linhas RF, DT, GB, RL; colunas A, B, C):\')\n    print(vals)\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--input\', type=Path, required=True, help=\'CSV oficial resumo_12_combinacoes.csv\')\n    p.add_argument(\'--output\', type=Path, default=Path(\'fig03_heatmap_exp07.png\'))\n    p.add_argument(\'--dpi\', type=int, default=350)\n    a = p.parse_args()\n    make_figure(a.input, a.output, a.dpi)\n', encoding='utf-8')
RESULTADO_FIG03 = FIG/'fig03_heatmap_exp07.png'
subprocess.run([sys.executable, str(SCRIPT_HEATMAP),
                '--input', str(ROOT/'resumo_12_combinacoes.csv'),
                '--output', str(RESULTADO_FIG03), '--dpi', '350'], check=True)
display(Image(filename=str(RESULTADO_FIG03)))
print('Figura 3 exportada em:', RESULTADO_FIG03)


## Figura 4 do artigo — sensibilidade `min_tx` (EXP-01J)

O diagrama científico usa o **CSV oficial EXP-01J**, e não valores simulados: `comparacao_elegibilidade.csv`, de 29/10/2020. O script verifica o SHA-256 do arquivo (8c855262…) e todos os quatro cenários.

- Painel esquerdo: quantidade de endereços elegíveis (**escala logarítmica explícita**).
- Painel direito: alertas sinalizados (contagem absoluta).
- Resultado complementar: **três pools conhecidos de referência** recuperados em todos os filtros.

**Atenção:** a comparação evidencia sensibilidade e cobertura, não precision/recall/F1 nem detecção comprovada de contratos inéditos. A Figura 4 final é reproduzida diretamente pelos dados reais e salva em `figures/fig04_sensibilidade_exp01j.png`.

In [ ]:
# Script idêntico ao versionado em scripts/figures/gerar_fig04_sensibilidade_exp01j.py
import subprocess, sys
from IPython.display import Image, display
SCRIPT_FIG04 = ROOT/'gerar_fig04_sensibilidade_exp01j.py'
SCRIPT_FIG04.write_text('#!/usr/bin/env python3\n"""Figura 4: EXP-01J — impacto de min_tx na cobertura e na contagem de alertas.\n\nFonte auditada: comparacao_elegibilidade.csv (EXP-01J, 17/11/2026)\nDrive: https://drive.google.com/file/d/1fGYC9pNVHlGILcEbJnlqBqulEWZTM0-o/view\n\nUso:\n  python scripts/figures/gerar_fig04_sensibilidade_exp01j.py \\\n    --input comparacao_elegibilidade.csv --output fig04_sensibilidade_exp01j.png\n\nNão calcula precision/recall/F1, nem presume que alertas restantes sejam corretos.\n"""\nfrom __future__ import annotations\nimport argparse\nfrom pathlib import Path\nimport hashlib\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import FuncFormatter, FixedLocator\nimport pandas as pd\n\nEXPECTED_SHA256 = \'8c855262196d5fcadb92ddf9f4c8acc2c8de55b9262ebaf609ffec99623d9b87\'\nEXPECTED_VALUES = {\n  1: (230658, 49, 3),\n  5: (7865, 14, 3),\n  10: (4038, 9, 3),\n  20: (2464, 7, 3),\n}\n\n\ndef fmt_br(v):\n    return f\'{int(v):,}\'.replace(\',\', \'.\')\n\n\ndef make_figure(input_path: Path, output_path: Path, dpi: int = 350):\n    raw = input_path.read_bytes()\n    checksum = hashlib.sha256(raw).hexdigest()\n    if checksum != EXPECTED_SHA256:\n        raise ValueError(f\'SHA-256 do CSV não confere: {checksum}\')\n    df = pd.read_csv(input_path).sort_values(\'min_tx\')\n    required = {\'min_tx\', \'enderecos_unicos\', \'alertas\', \'pools_artigo_recuperados\'}\n    if not required.issubset(df.columns):\n        raise ValueError(f\'Colunas ausentes: {required - set(df.columns)}\')\n    if len(df) != 4:\n        raise ValueError(\'Esperados quatro cenários EXP-01J.\')\n    got = {int(r.min_tx): (int(r.enderecos_unicos),int(r.alertas),int(r.pools_artigo_recuperados))\n           for r in df.itertuples()}\n    if got != EXPECTED_VALUES:\n        raise ValueError(f\'Valores diferentes dos quatro cenários auditados: {got}\')\n    if not (df.janelas_elegiveis.equals(df.enderecos_unicos) and\n            df.janelas_classificadas.equals(df.enderecos_unicos) and\n            df.janelas_invalidadas_por_feature.eq(0).all()):\n        raise ValueError(\'Cobertura da classificação diferente da auditoria EXP-01J.\')\n\n    mins = df.min_tx.to_numpy()\n    n_eligible = df.enderecos_unicos.to_numpy()\n    n_alerts = df.alertas.to_numpy()\n\n    plt.rcParams.update({\'font.family\': \'DejaVu Sans\', \'font.size\': 10,\n                         \'axes.spines.top\': False, \'axes.spines.right\': False})\n    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10.5, 4.0), facecolor=\'white\')\n    fig.subplots_adjust(left=.095, right=.975, top=.77, bottom=.30, wspace=.31)\n    # Os dois painéis usam apenas valores de fonte, com escalas diferentes explicitamente.\n    ax = axes[0]\n    ax.plot(mins, n_eligible, \'o-\', linewidth=2.1, markersize=7.5)\n    ax.set_yscale(\'log\')\n    ax.set_ylim(1500, 450000)\n    ax.set_xlim(.4, 20.6)\n    ax.set_xticks(mins)\n    ax.yaxis.set_major_locator(FixedLocator([2000, 10000, 100000]))\n    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: fmt_br(v)))\n    ax.minorticks_off()\n    ax.set_xlabel(\'Mínimo de transações ($min_{tx}$)\')\n    ax.set_ylabel(\'Endereços elegíveis (escala log)\')\n    ax.set_title(\'(a) Cobertura da classificação\', loc=\'left\', fontsize=11.5, pad=11)\n    ax.grid(axis=\'y\', linewidth=.7, alpha=.22)\n    ax.axvline(10, linewidth=1, linestyle=\'--\', alpha=.35)\n    for x,y in zip(mins,n_eligible):\n        ax.annotate(fmt_br(y), (x,y), xytext=(0, 10 if x!=5 else -17),\n                    textcoords=\'offset points\', ha=\'center\', va=\'bottom\', fontsize=9.7,\n                    weight=\'semibold\')\n\n    ax = axes[1]\n    ax.plot(mins, n_alerts, \'o-\', linewidth=2.1, markersize=7.5)\n    ax.set_xlim(.4,20.6)\n    ax.set_ylim(0,58)\n    ax.set_xticks(mins)\n    ax.set_yticks([0,10,20,30,40,50])\n    ax.set_xlabel(\'Mínimo de transações ($min_{tx}$)\')\n    ax.set_ylabel(\'Contratos sinalizados\')\n    ax.set_title(\'(b) Alertas produzidos\', loc=\'left\', fontsize=11.5, pad=11)\n    ax.grid(axis=\'y\', linewidth=.7, alpha=.22)\n    ax.axvline(10, linewidth=1, linestyle=\'--\', alpha=.35)\n    for x,y in zip(mins,n_alerts):\n        ax.annotate(str(y),(x,y),xytext=(0,10),textcoords=\'offset points\',\n                    ha=\'center\',fontsize=11,weight=\'bold\')\n\n    fig.text(.095,.925,\'EXP-01J | Sensibilidade ao filtro mínimo de transações\',\n             color=\'#202E3A\',fontsize=15,weight=\'bold\')\n    fig.text(.095,.85,\'Ethereum · 29/10/2020 · mesmo modelo e limiar de decisão (0,5)\',\n             color=\'#5D6A79\',fontsize=10.5)\n    fig.text(.095,.17,\'Os 3 pools Tornado Cash de referência foram sinalizados nos quatro cenários.\',\n             color=\'#333D47\',fontsize=9.7)\n    fig.text(.095,.095,\'O filtro altera a população avaliada; redução de alertas não demonstra aumento de precisão.\',\n             color=\'#5D6A79\',fontsize=9.3)\n    output_path.parent.mkdir(parents=True,exist_ok=True)\n    fig.savefig(output_path,dpi=dpi,bbox_inches=\'tight\',pad_inches=.12,facecolor=\'white\')\n    plt.close(fig)\n    print(\'PNG:\',output_path.resolve())\n    print(\'SHA-256 do CSV de entrada:\',checksum)\n    print(\'Cenários verificados:\',got)\n    return got\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--input\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,default=Path(\'fig04_sensibilidade_exp01j.png\'))\n    p.add_argument(\'--dpi\',type=int,default=350)\n    args=p.parse_args()\n    make_figure(args.input,args.output,args.dpi)\n', encoding='utf-8')
PNG_FIG04 = FIG/'fig04_sensibilidade_exp01j.png'
subprocess.run([sys.executable, str(SCRIPT_FIG04), '--input',
                str(ROOT/'comparacao_elegibilidade.csv'), '--output', str(PNG_FIG04),
                '--dpi', '350'], check=True)
display(Image(filename=str(PNG_FIG04)))
print('Figura 4 reproduzida:', PNG_FIG04)


## 4. Distribuição das características e dispersão 2D do treino

As figuras abaixo mostram a **matriz original** e ilustram diferenças descritivas. **Atenção:** 23 linhas positivas correspondem a 1 contrato; 100 negativas a 100 contratos. O gráfico PCA é uma projeção geométrica, não uma rede de transações, não é generalização e não implica agrupamentos de atividades ilícitas.

**Figura 2 do artigo:** Esta PCA usa as cinco características da matriz rotulada original, após padronização. Os 23 exemplos positivos correspondem a um único contrato observado em 23 dias. Os grupos são os **rótulos de treinamento**, não alertas de 2026. A PCA é descritiva e não treina um detector, não é UMAP/GNN, nem comprova separação generalizável. O PNG é salvo em `figures/fig02_pca_exp07.png` e pode ser inserido em `figure*` no Overleaf.


In [ ]:
D=pd.read_parquet(ROOT/'features_treino_original.parquet')
features=['n_tx','value_entropy','fee_entropy','internal_value_entropy','call_count_mean']
assert set(features+['label','recipient']).issubset(D.columns)
assert D.loc[D.label==1,'recipient'].nunique()==1
for feature in ['value_entropy','fee_entropy','internal_value_entropy','call_count_mean']:
   data=[D.loc[D.label==0,feature].astype(float).to_numpy(),D.loc[D.label==1,feature].astype(float).to_numpy()]
   fig,ax=plt.subplots(figsize=(6.5,3.9));parts=ax.boxplot(data,tick_labels=['100 negativos\n(100 contratos)','23 positivos\n(1 contrato)'],patch_artist=True,showfliers=True)
   for patch,color in zip(parts['boxes'],[GRAY,PETROL]):patch.set_facecolor(color);patch.set_alpha(.55)
   ax.set_title('Distribuição de '+feature);ax.set_ylabel(feature)
   export('04_boxplot_'+feature+'.png')
# Figura 2 do artigo — PCA exploratória calculada DIRETAMENTE do Parquet real
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display
import hashlib
expected_sha = '0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32'
assert sha(ROOT/'features_treino_original.parquet') == expected_sha, 'Hash da matriz diverge do original auditado'
assert len(D)==123 and D.label.value_counts().to_dict()=={0:100,1:23}
assert D.loc[D.label==0,'recipient'].nunique()==100
assert D.loc[D.label==1,'recipient'].nunique()==1
assert not D[features].isna().any().any()
X = StandardScaler().fit_transform(D[features].astype(float))
pca = PCA(n_components=2, svd_solver='full')
proj = pca.fit_transform(X)
explained = pca.explained_variance_ratio_ * 100
mask = D.label.to_numpy()==1
fig,ax=plt.subplots(figsize=(8.7,4.8))
fig.patch.set_facecolor('white')
fig.subplots_adjust(left=.095,right=.965,top=.88,bottom=.22)
cycle=plt.rcParams['axes.prop_cycle'].by_key()['color']
ax.scatter(proj[~mask,0],proj[~mask,1],s=39,c=cycle[0],alpha=.75,
           edgecolors='white',linewidths=.35,marker='o',
           label='Não mixer: 100 observações / 100 contratos',zorder=2)
ax.scatter(proj[mask,0],proj[mask,1],s=60,c=cycle[1],alpha=.95,
           edgecolors='white',linewidths=.65,marker='D',
           label='Tornado Cash: 23 dias / 1 contrato',zorder=3)
ax.axhline(0,color='#D3DCE4',lw=.8,zorder=0)
ax.axvline(0,color='#D3DCE4',lw=.8,zorder=0)
ax.grid(alpha=.12,linestyle='--',linewidth=.65,zorder=0)
ax.set_axisbelow(True)
ax.set(xlabel=f'Componente principal 1 ({explained[0]:.1f}% da variância)',
       ylabel=f'Componente principal 2 ({explained[1]:.1f}% da variância)')
ax.set_title('Distribuição dos exemplos rotulados — EXP-07',loc='left',fontsize=13,
             weight='semibold',pad=14)
ax.legend(loc='upper right',frameon=True,fancybox=False,fontsize=9.2,
          facecolor='white',edgecolor='#D5DEE7')
fig.text(.10,.095,'Projeção exploratória das 5 características padronizadas. '
         'As 23 observações positivas são do mesmo contrato.',fontsize=9.2,color='#485665')
fig.text(.10,.057,'Não representa validação independente, detecção de anomalias '
         'ou grafo de transações.',fontsize=9,color='#485665')
path_pca = FIG/'fig02_pca_exp07.png'
fig.savefig(path_pca,dpi=350,facecolor='white')
display(fig)
plt.close(fig)
print('Figura 2 salva:',path_pca)
print(f'PCA explicou {explained.sum():.2f}% da variância nas duas componentes')


## 5. Visualização dos alertas de janeiro de 2026

As nove pontuações são do arquivo `sinalizados_teste_2026.csv`. Os seis contratos sem rótulo de referência são **indeterminados**, não negativos. O scatter plot é de **atributos estatísticos**, e não um grafo real de transferências.

In [ ]:
A=pd.read_csv(ROOT/'sinalizados_teste_2026.csv')
assert len(A)==9
is_ref=A['rotulo'].fillna('').isin(['0.1 ETH','1 ETH','10 ETH'])
assert int(is_ref.sum())==3
Z=A.sort_values('prob',ascending=True).copy();ref=Z['rotulo'].fillna('').isin(['0.1 ETH','1 ETH','10 ETH'])
fig,ax=plt.subplots(figsize=(8,5.3));y=np.arange(len(Z))
ax.hlines(y,0,Z.prob,color=np.where(ref,PETROL,GRAY),lw=3,alpha=.7)
ax.scatter(Z.prob,y,c=np.where(ref,PETROL,GRAY),s=95,zorder=3)
ax.set_yticks(y,[f'{"Tornado Cash" if r else "Indeterminado"} · {a[:8]}…' for a,r in zip(Z.recipient,ref)])
ax.set_xlim(.0,1.1);ax.set(xlabel='Pontuação produzida pelo classificador (não é prova)',title='Janeiro/2026 — escores dos nove alertas')
ax.axvline(.85,ls='--',color=ORANGE,label='Limiar congelado 0,85');ax.legend(fontsize=9)
export('06_alertas_2026_lollipop.png')
fig,ax=plt.subplots(figsize=(8,5.2))
for v,col,lab in [(True,PETROL,'Três contratos conhecidos'),(False,GRAY,'Seis indeterminados')]:
 m=is_ref==v;ax.scatter(A.loc[m,'fee_entropy'],A.loc[m,'call_count_mean'],s=35+2*A.loc[m,'n_tx'],alpha=.78,c=col,label=lab,edgecolors='white',linewidths=.65)
ax.set_yscale('log');ax.set(xlabel='fee_entropy',ylabel='call_count_mean (escala log)',title='Atributos dos alertas — visualização exploratória')
ax.legend(fontsize=9);export('07_dispersao_atributos_alertas.png')

## 6. Matriz de confusão agregada dos cinco folds (EXP-07)

Os registros por fold já foram auditados. A agregação a seguir soma as predições internas, sem tratá-las como evidência de generalização. Esta visualização não é a matriz de confusão da população Ethereum.

In [ ]:
F=pd.read_csv(ROOT/'metricas_12_combinacoes_por_fold.csv')
r=F[(F.configuracao=='A_original_5')&(F.modelo=='RandomForest')]
assert len(r)==5
cm=np.array([[r.tn.sum(),r.fp.sum()],[r.fn.sum(),r.tp.sum()]],dtype=int)
fig,ax=plt.subplots(figsize=(6,4.5));im=ax.imshow(cm,cmap='Blues')
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Classe 0','Classe 1'],yticklabels=['Classe 0','Classe 1'],xlabel='Predito',ylabel='Rótulo da matriz',title='Confusão agregada — 5 folds internos')
for i in range(2):
 for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontweight='bold',color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=18)
fig.colorbar(im,ax=ax);export('08_confusao_interna_rf.png')
print('AVISO: contrato positivo é o mesmo nos cinco folds. NUNCA apresentar como teste em novo protocolo.')

## 7. Opcional — reexecutar 12 classificadores com a matriz real

Não é necessário para gerar as figuras. Para auditoria dos modelos, execute [EXP-07 v2 original](https://drive.google.com/file/d/1nKI1sM5pbY-UOAI6nxraTTSzkNFY8MHW/view), que contém todo o protocolo pré-registrado, hashes e gravação em Drive. O notebook integrado é o **painel de estudo e apresentação**; o EXP-07 v2 original continua como registro autoritativo de execução.

## 8. Exportar gráficos para seu Drive (opcional)

`EXPORTAR_PARA_DRIVE=False` por padrão: a demonstração é segura contra criação de pastas duplicadas. Ao ativar, o Colab cria uma pasta privada no Drive e envia os PNGs. Os gráficos continuam na pasta local `/content/lab_ethereum/figures/` mesmo sem exportação.

In [ ]:
EXPORTAR_PARA_DRIVE=False
if EXPORTAR_PARA_DRIVE:
    name='LAB_GRAFICOS_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    folder=service.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':['1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI']},fields='id,name').execute()
    for p in sorted(FIG.glob('*.png')):
        media=MediaFileUpload(str(p),mimetype='image/png',resumable=True)
        service.files().create(body={'name':p.name,'parents':[folder['id']]},media_body=media,fields='id').execute()
    print('Gráficos salvos no Drive na pasta de resultados:',folder['name'],folder['id'])
else:print('Gráficos disponíveis para inspeção neste Colab:',FIG)